# Train a CNN on Shapes

In [1]:
%load_ext autoreload
%autoreload 2

In [23]:
from pathlib import Path
import zipfile

import numpy as np
import matplotlib.pyplot as plt

import torch

import torchvision.transforms as transforms
from torch.utils.data import random_split, DataLoader

from src.utils import load_params, extract_zipfile, resize_image, make_shape_dataset
from src.networks import CustomDataset, CNNNet, calculate_accuracy

In [24]:
# Paths and directories
ROOT = Path().cwd().parent
data_dir = ROOT / "data"
fig_dir = ROOT / "figures"
model_dir = data_dir / "models"
shapes_dir = data_dir / "stimuli" / "shape_dataset"

zip_file = shapes_dir / "shape_dataset.zip"

ext_dir = shapes_dir / "output"
resize_dir = shapes_dir / "resized"
resize_dir.mkdir(parents=True, exist_ok=True)
new_img_size = 100

In [4]:
# Unzip the dataset if needed
extract_zipfile(zip_file, shapes_dir)

In [5]:
# Resize images
for img in ext_dir.iterdir():

    resize_name = resize_dir / f"{img.stem}_resized.png"
    resize_image(image_path=img, new_size=(new_img_size, new_img_size), output_path=resize_name)

In [31]:
# Create the dataset with labels
df = make_shape_dataset(images_dir=resize_dir, save_dir=shapes_dir)

In [32]:
df.head()

,FILE_NAME,LABEL
0,Circle_000dfc5c-2a92-11ea-8123-8363a7ec19e6_re...,0
1,Circle_000ed2d8-2a8a-11ea-8123-8363a7ec19e6_re...,0
2,Circle_0013f29e-2a9a-11ea-8123-8363a7ec19e6_re...,0
3,Circle_001d7284-2a85-11ea-8123-8363a7ec19e6_re...,0
4,Circle_001de166-2a89-11ea-8123-8363a7ec19e6_re...,0


In [33]:
# Make training set
train_dataset = CustomDataset(
    data_frame=df,
    root_dir=resize_dir,
    transform=transforms.ToTensor()
)

batch_size = 500

train_set, test_set = random_split(train_dataset, [72000, 18000])
train_loader = DataLoader(dataset=train_set, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_set, batch_size=batch_size, shuffle=True)

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = CNNNet().to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = (1e-3))

training_cost_array = []
testing_cost_array = []
training_accuracy_array = []
testing_accuracy_array = []
epoch_array = []
n_epochs = 25

for epoch in range(1, n_epochs + 1):
    training_losses = []
    testing_losses = []

    for batch_idx, (train_data, train_targets) in enumerate(train_loader):
        train_data = train_data.to(device=device)
        train_targets = train_targets.to(device=device)
        training_score, _ = model(train_data)
        training_loss = criterion(training_score, train_targets)
        training_losses.append(training_loss.item())
        optimizer.zero_grad()
        training_loss.backward()
        optimizer.step()

    with torch.no_grad():
        for test_data, test_target in test_loader:
            test_data = test_data.to(device=device)
            test_target = test_target.to(device=device)
            testing_score, _ = model(test_data)
            testing_loss = criterion(testing_score, test_target)  # sum up batch loss
            testing_losses.append(testing_loss.item())

    train_accuracy = calculate_accuracy(train_loader, model, device=device)
    test_accuracy = calculate_accuracy(test_loader, model, device=device)

    printable_training_loss = sum(training_losses)/len(training_losses)
    printable_testing_loss = sum(testing_losses)/len(testing_losses)

    training_cost_array.append(printable_training_loss)
    testing_cost_array.append(printable_testing_loss)
    training_accuracy_array.append(train_accuracy)
    testing_accuracy_array.append(test_accuracy)
    epoch_array.append(epoch)
    print(f'| Epoch: {epoch} |  Training Cost: {round(printable_training_loss, 3)} | Testing Cost: {round(printable_testing_loss, 3)} | Training Accuracy: {round(train_accuracy, 3)}% | Testing Accuracy: {round(test_accuracy, 3)}% |')

In [ ]:
# Save model
model_path = model_dir / "trained_CNN_128.pt"
torch.save(model.state_dict(), model_path)

In [ ]:
# Plot accuracy
fig, axes = plt.figure(1, 2, figsize=(30, 10))

axes[0].plot(epoch_array, training_cost_array, color="red", label="Training")
axes[0].plot(epoch_array, testing_cost_array, color = 'blue', label = "Testing")
axes[0].title("Variation of Cost with Epoch")
axes[0].xlabel("Epoch")
axes[0].ylabel("Cost")

axes[1].plot(epoch_array, training_accuracy_array, color='red', label="Training")
axes[1].plot(epoch_array, testing_accuracy_array, color = 'blue', label = "Testing")
axes[1].title("Variation of Accuracy with Epoch")
axes[1].xlabel("Epoch")
axes[1].ylabel("Accuracy")

plt.legend()
plt.show()

fig.savefig(fig_dir / "CNN_training.png")